# 🔐 **Security Evaluation of a Face Recognition System** 

## Transferability of Adversarial Examples on NN2

**Academic Year:** 2024-2025  
**Group:** 04

---

### 👥 Team Members
- **Agostino Cardamone** — `0622702276`
- **Asja Antonucci**     — `0622702437`
- **Chiara Ferraioli**   — `0622702169`

---

### 📚 Overview of This Section

1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Evaluation on Alternative Model](#2-evaluation-on-alternative-model)
3. [Attack Transferability Analysis](#3-attack-transferability-analysis)

## 1. Setup and Data Loading

#### Environment Setup

To ensure reproducibility and avoid package conflicts, it is strongly recommended to run all experiments in an isolated environment. We use Conda to create and manage the project environment, and all Python dependencies are listed in the requirements.txt file.

In [ ]:
# 1) Create a new environment named “aic_env”
#conda create -n aic_env python=3.10 -y

# 2) Switch into the new environment
# On Windows:
# conda activate aic_env
# On Linux/macOS:
# source activate aic_env

# 3) Install all dependencies
#!pip install -r requirements.txt

import os 

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
print("torch.version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

#### Dataset Configuration and Paths

This section defines the core paths and settings used throughout the project to manage the dataset structure and preprocessing behavior.

- `dataset_dir` points to the root directory containing the dataset files.
- `dataset_selection` controls whether to regenerate the test set from scratch (⚠️set to True only if you have extracted `vggface2_train` inside `dataset/vggface2_train/trainset`⚠️)
- `mtcnn_processing_nn1` determines whether to apply MTCNN face alignment for NN1 preprocessing.
- `test_set_rnd` specifies whether the test set should be built randomly or from the pre-defined class list in `test_set.csv`.

Metadata and folder structure:
- `vgg2_dataset_annotations_path` points to `identity_meta.csv`, which contains class metadata (name, gender, etc.).
- `test_set_data_folder` is the location of test samples.
- `test_set_annotations_folder` contains the CSV file describing the test set structure.

All experiment outputs will be saved to:
- `results_folder` — for accuracy results and evaluations.
- `adversarial_folder` — for storing generated adversarial images.

The variable `device` automatically selects GPU if available, otherwise defaults to CPU.

In [ ]:
from utils import *         # Project-specific utilities and imports

# —————————————————————————————————————————————
#           Paths and configuration
# —————————————————————————————————————————————

# Base directory containing all dataset-related files
dataset_dir = os.path.join(os.getcwd(), 'dataset')

# If True, a new test set will be built by sampling and copying images from the original VGGFace2 dataset
# NOTE: This requires the dataset to be downloaded and extracted under 'vggface2_train/trainset'
# If False, the existing CSV files will be loaded without modifying or copying any images
dataset_selection = False  

# If True, test images will be aligned and cropped using MTCNN preprocessing (for NN1 compatibility)
mtcnn_processing_nn1 = False

# If True, the test set will be built via random sampling of identities and images
# If False, the test set will be built based on predefined class IDs listed in 'test_set.csv'
test_set_rnd = False

# Path to VGGFace2 identity metadata (includes Class_ID, Name, Gender, etc.)
vgg2_dataset_annotations_path = os.path.join(dataset_dir, 'identity_meta.csv')

# Folder structure for test set files and images
test_set_folder              = os.path.join(dataset_dir, 'testset')
test_set_data_folder         = os.path.join(test_set_folder, 'samples')
test_set_annotations_folder  = os.path.join(test_set_folder, 'test_set.csv')

# Directory to store evaluation results (e.g., SEC curves)
results_folder = os.path.join(os.getcwd(), 'results')

# Directory to save generated adversarial examples
adversarial_folder = os.path.join(os.getcwd(), 'attacks')

# Device configuration: use GPU if available, otherwise fallback to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


#### Load Test Set and Class Labels

This block performs two critical initializations:

1. **Load test set metadata**
   - The file `test_set.csv` is read into a DataFrame.
   - It contains exactly 100 test identities, each with 10 face images located in `testset/samples/`.
   - The total number of expected images is computed as `100 × 10 = 1000`.

2. **Load class label mappings**
   - The face recognition model requires access to the full list of class names (8631 identities).
   - These are loaded from a `.npy` file originally published by the official [`rcmalli/keras-vggface`](https://github.com/rcmalli/keras-vggface) repository.
   - If the file is not present locally, it is automatically downloaded.
   - Any surrounding whitespace in label strings is stripped to ensure clean formatting.

This step is necessary to convert model outputs (class indices) into readable identity labels.

In [ ]:
# —————————————————————————————————————————————
#              Load annotation CSVs
# —————————————————————————————————————————————

# Read the existing test_set.csv describing our 100 test identities
# (each identity will have 10 samples in the folder structure)
test_set = pd.read_csv(
    filepath_or_buffer=test_set_annotations_folder,
    sep=',',
    skipinitialspace=True,
    engine='python'
)

# Calculate how many total images we expect in the test set:
# number of identities × 10 images each
test_set_size = len(test_set) * 10

# ————————————————————————————————————————————
#  Download and load the class‐label mapping
# ————————————————————————————————————————————

# Our face‐recognition model expects a NumPy array of all 8631 labels.
# We download it from the official rcmalli/keras-vggface repo if missing.
labels_url = (
    "https://github.com/rcmalli/keras-vggface/"
    "releases/download/v2.0/rcmalli_vggface_labels_v2.npy"
)
labels_path = os.path.join(dataset_dir, 'rcmalli_vggface_labels_v2.npy')

# Ensure the directory for labels_path exists
os.makedirs(os.path.dirname(labels_path), exist_ok=True)

# Download only if the file does not already exist on disk
if not os.path.exists(labels_path):
    print(f"Downloading LABELS to {labels_path}…")
    urllib.request.urlretrieve(labels_url, labels_path)

# Load the .npy file into a NumPy array and strip any padding whitespace
LABELS = np.load(labels_path)
LABELS = np.char.strip(LABELS)

In this section, we prepare the test set so it can be efficiently used during inference. The aim is to create a `DataLoader` that iterates over face images, optionally applying preprocessing steps, and associates each image with the correct identity label.

The process includes the following components:

- **Custom Collate Function**  
  A lightweight `collate_fn` is defined to simplify how batches are constructed. Since face detection and alignment may occur outside the `DataLoader` logic, this function ensures that only individual samples are returned without standard batching behaviour.

- **Image Transformations**  
  A basic image preprocessing pipeline is defined using `torchvision.transforms`. If MTCNN alignment is not enabled, each image is resized to 160×160 pixels (the input size required by the face recognition model) and then converted to a tensor.

- **Dataset Definition via ImageFolder**  
  The test images are loaded using `ImageFolder`, which expects the directory structure to be organised such that each identity has its own folder. The dataset automatically assigns a numerical label to each subfolder and loads all images accordingly.

- **Label Mapping**  
  A custom mapping `idx_to_class` is constructed to associate the internal numeric labels used by `ImageFolder` with the actual identity names provided in the test set metadata. This ensures label consistency throughout the evaluation process.

- **Final DataLoader**  
  The dataset is wrapped in a `DataLoader` for iteration. No parallel workers (`num_workers=0`) are used to maintain compatibility and simplicity. This `DataLoader` can now be used to either:
  - Apply face detection and alignment via MTCNN (if enabled), or
  - Pass pre-aligned images directly to the recognition model.

This structure ensures that the test set is handled in a clean, modular, and reproducible way.

In [ ]:

# Define the collate function for the DataLoader
def collate_fn(x):
    return x[0]

# Define the transforms for the DataLoader
transforms = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

# Create a DataLoader for the test set by using the ImageFolder dataset
dataset = datasets.ImageFolder(root=test_set_data_folder, transform=transforms if not mtcnn_processing_nn1 else None)
dataset.idx_to_class = {i: c.replace('', '') for i, c in enumerate(test_set['Name'])}
dataloader = DataLoader(dataset, collate_fn=collate_fn, num_workers=0)                                                                  

The aligned dataset, which was stored in a PyTorch `DataLoader`, includes both the face images and their corresponding identity labels. Once loaded, the individual samples are unpacked and converted into two tensors: one containing all the aligned images, and the other containing the associated class labels.

To ensure compatibility with downstream processing steps, the image tensor is converted from PyTorch format to a NumPy array. This is especially useful when the data needs to be passed into frameworks such as **ART (Adversarial Robustness Toolbox)**, which often expect NumPy inputs for generating adversarial examples.

Finally, the script reconstructs a reverse label mapping that allows us to convert class names (i.e., the folder names used in `ImageFolder`) back into numerical indices. This is essential for consistent evaluation and interpretation of results, especially when comparing predicted and true labels.


In [ ]:
# Load the aligned DataLoader
dataloader_aligned_nn1 = torch.load('dataset/dataloader_aligned_nn1.pt', weights_only=False)

# Get the x_test_aligned_nn1 and y_test_aligned_nn1 from the DataLoader
x_test_aligned_nn1, y_test_aligned_nn1 = zip(*[(sample[0], sample[1]) for sample in dataloader_aligned_nn1])

# Create the batches of the tensors of the images and labels
x_test_aligned_nn1 = torch.stack(x_test_aligned_nn1)
y_test_aligned_nn1 = torch.tensor(y_test_aligned_nn1)

x_test_aligned_nn1 = x_test_aligned_nn1.cpu().numpy() 

# Inverti il dizionario per cercare per valore
class_to_idx = {v: k for k, v in dataset.idx_to_class.items()}

selected_classes = ['Andrea_Bocelli','Gigi_DAlessio','Diego_Abatantuono','Diego_Maradona','Francesco_Totti','Dries_Mertens']

idx_test_images = []
for name in selected_classes:
    if name in class_to_idx:
        idx_test_images.append(class_to_idx[name])
    else:
        print(f"Name not found: {name}")

#### Load and Prepare Face Recognition Model (NN1)

This section sets up the face recognition model referred to as **NN1**, based on the `InceptionResnetV1` architecture provided by the `facenet-pytorch` library.

- The model is initialised with weights pre-trained on the **VGGFace2** dataset, which contains over 8,000 people identities.
- It is set to evaluation mode (`.eval()`), disabling stochastic layers such as dropout and batch normalisation updates, ensuring deterministic inference.
- By default, the model outputs a 512-dimensional feature embedding for each face. However, enabling the `.classify = True` flag appends a classification head, allowing the model to directly output class logits over the **8,631 identities** present in the VGGFace2 training set.

In [ ]:
from facenet_pytorch import InceptionResnetV1

# ——————————————————————————————————————————————————————
#   Initialize the pre-trained face-recognition model
# ——————————————————————————————————————————————————————

# We use the InceptionResnetV1 architecture from the facenet-pytorch package,
# pre-trained on the VGGFace2 dataset for high-quality face embeddings.
# By calling .eval(), we set the model to inference mode (disables dropout, batchnorm updates).
# We then move the model to the appropriate device (GPU if available, else CPU).
nn1 = InceptionResnetV1(
    pretrained='vggface2'  # load weights trained on the VGGFace2 face dataset
).eval().to(device)         # switch to evaluation mode and transfer to GPU/CPU

# —————————————————————————————————————————————
#           Enable classification head
# —————————————————————————————————————————————

# By default, InceptionResnetV1 returns 512-dimensional embeddings.
# Setting .classify instructs the model to append a linear classification
# layer on top of the embeddings, so that nn1(input) returns raw class logits
# for all identities in VGGFace2 (8 631 classes), instead of embeddings.
nn1.classify = True

## 2. Evaluation on Alternative Model

The alternative model used for cross-evaluation in this project is based on the well-established `ResNet-50` architecture. `ResNet`, or `Residual Network`, is a deep convolutional neural network that introduced the concept of residual learning to address the vanishing gradient problem in very deep networks.

The implementation constructs a `50-layer deep ResNet` by stacking several `Bottleneck blocks`, which are a specialised form of residual block designed to maintain efficiency while maintaining computational efficiency. Each block contains `three convolutional layers` (`1×1`, `3×3`, and `1×1`), and incorporates `skip connections` that add the input of the block directly to its output. This identity mapping enables the model to learn residual functions and accelerates convergence.

The `include_top` parameter determines whether the classification head is included. In this evaluation, `include_top=True` is used to retain the final fully connected layer for identity prediction.

Weight initialisation is performed using He normal initialisation for convolutional layers, and batch normalisation layers are initialised with scale=1 and bias=0. This helps ensure stable training and consistent performance.

In [ ]:
from NN2.models.resnet import *
from NN2.utils import *
from NN2.datasets import *
from utils import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize the ResNet50 model
nn2 = resnet50(num_classes=8631, include_top=True)
id_label_dict = get_id_label_map("dataset\identity_meta.csv")

# Load the weights of the model
load_state_dict(nn2, 'NN2/resnet50_scratch_weight.pkl')
nn2 = nn2.eval().to(device)

This block defines the main paths and runtime settings for the evaluation. It specifies where the dataset, annotations, and results are stored, and how the test set is constructed — either via random sampling or from a predefined CSV.

It also sets flags for `MTCNN preprocessing` and establishes the working device (GPU if available, otherwise CPU).

In [ ]:
# —————————————————————————————————————————————
#           Paths and configuration
# —————————————————————————————————————————————

# Base folder for all dataset files
dataset_dir = os.getcwd() + '/dataset'

mtcnn_processing_nn2 = True

# If True, the test set will be build through random sampling.
# If False, the test set will be build using the classes specified in the already existing test_set.csv
test_set_rnd = False

# Full VGGFace2 metadata CSV (Class_ID, Name, Gender, etc.)
vgg2_dataset_annotations_path = os.path.join(dataset_dir, 'identity_meta.csv')

# Folder structure for test set
test_set_folder              = os.path.join(dataset_dir, 'testset')
test_set_data_folder         = os.path.join(test_set_folder, 'samples')
test_set_annotations_folder  = os.path.join(test_set_folder, 'test_set.csv')

results_folder = os.getcwd() + '/results'
adversarial_folder = os.getcwd() + '/attacks'

# Compute device (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Now we load the annotation file `test_set.csv`, which, as before, defines the structure of the evaluation set used throughout the experiments. As done previously, the code manages the label mapping required by the face recognition model. The model expects identity labels in a specific order defined by the official `VGGFace2` release.

In [ ]:
# —————————————————————————————————————————————
#              Load annotation CSVs
# —————————————————————————————————————————————

# Read the existing test_set.csv describing our 100 test identities
# (each identity will have 10 samples in the folder structure)
test_set = pd.read_csv(
    filepath_or_buffer=test_set_annotations_folder,
    sep=',',
    skipinitialspace=True,
    engine='python'
)

# Calculate how many total images we expect in the test set:
# number of identities × 10 images each
test_set_size = len(test_set) * 10

# ————————————————————————————————————————————
#  Download and load the class‐label mapping
# ————————————————————————————————————————————

# Our face‐recognition model expects a NumPy array of all 8631 labels.
# We download it from the official rcmalli/keras-vggface repo if missing.
labels_url = (
    "https://github.com/rcmalli/keras-vggface/"
    "releases/download/v2.0/rcmalli_vggface_labels_v2.npy"
)
labels_path = os.path.join(dataset_dir, 'rcmalli_vggface_labels_v2.npy')

# Ensure the directory for labels_path exists
os.makedirs(os.path.dirname(labels_path), exist_ok=True)

# Download only if the file does not already exist on disk
if not os.path.exists(labels_path):
    print(f"Downloading LABELS to {labels_path}…")
    urllib.request.urlretrieve(labels_url, labels_path)

# Load the .npy file into a NumPy array and strip any padding whitespace
LABELS = np.load(labels_path)
LABELS = np.char.strip(LABELS)

As done previously, this section initialises the `MTCNN face detector` to perform face detection, cropping, and alignment in a single step. The core configuration remains consistent with earlier usage, with one key difference: the `image_size` parameter is now set to `224`. This adjustment ensures that the resulting face crops match the input dimensions expected by the `NN2 model (ResNet-50)`, which operates on images of size `224×224`.

All other parameters remain unchanged to preserve detection reliability and alignment consistency. This setup ensures seamless integration between the preprocessing stage and the architecture of the alternative model.

In [ ]:
# ———————————————————————————————————————————————————
#  Initialize the face detector and aligner (MTCNN)
# ———————————————————————————————————————————————————
# We use MTCNN from facenet-pytorch to detect, crop, and align faces in one step.
# When you call face_detector_nn2(img_batch), it returns a tensor of shape [B, 3, image_size, image_size]
# containing the aligned face crops, ready to feed into nn1 or an adversarial attack.

face_detector_nn2 = MTCNN(
    image_size=224,                             # int: output height/width of each face crop (default=160)
    margin=0,                                   # int: number of pixels to expand the face bounding box (default=0)
    min_face_size=20,                           # int: minimum face size (in pixels) that the detector will attempt to locate (default=20)
    thresholds=[0.6, 0.7, 0.7],                 # list of 3 floats: score thresholds for each detection stage—
                                                #   P-Net, R-Net, and O-Net respectively (default=[0.6, 0.7, 0.7])
    factor=0.709,                               # float: scale factor between pyramid levels; controls the search granularity (default=0.709)
    post_process=True,                          # bool: whether to apply face alignment post-processing (True)
    select_largest=True,                        # bool: if multiple faces are detected, return only the largest one (True)
    selection_method="center_weighted_size",    # str: heuristic for choosing among multiple detections—
                                                #   options include "largest" or "center_weighted_size" (default="center_weighted_size")
    keep_all=False,                             # bool: if True, return all detected faces; if False, return only one (default=False)
    device=device                               # torch.device or str: computation device, e.g. "cuda:0" or "cpu"
)

Following the same approach used for NN1, this section defines a custom `collate_fn` for the `DataLoader` and sets up the image preprocessing pipeline. The main difference here is that the `Resize transformation` is set to `224×224`, which matches the input resolution expected by the NN2 (ResNet-50) model.

The dataset is loaded using `ImageFolder`, and transformations are applied only when MTCNN preprocessing is not used. Identity names are mapped to class indices using the test set annotations. Aside from the adjusted image size, the rest of the setup mirrors the configuration adopted for the NN1 evaluation.

In [ ]:
# Define the collate function for the DataLoader
def collate_fn(x):
    return x[0]

# Define the transforms for the DataLoader
transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Create a DataLoader for the test set by using the ImageFolder dataset
dataset = datasets.ImageFolder(root=test_set_data_folder, transform=transforms if not mtcnn_processing_nn2 else None)
dataset.idx_to_class = {i: c.replace('', '') for i, c in enumerate(test_set['Name'])}
dataloader = DataLoader(dataset, collate_fn=collate_fn, num_workers=0)                                                                  

As done earlier, this section handles the face cropping and alignment process by checking for the existence of a folder containing already processed images. If the folder with cropped faces (`cropped_faces_nn2`) is not present or is empty, the script proceeds to perform face detection and alignment using the configured MTCNN detector.

For each image in the dataset, the corresponding class directory is created (if not already present), and the aligned face is saved to disk. The images and their labels are collected into tensors — `x_test_aligned_nn2` for the processed faces and `y_test_aligned_nn2` for their corresponding labels.

Once all images have been processed, these tensors are saved for later use. Additionally, a new `DataLoader` is created containing the aligned test set, mirroring the same structure and logic adopted previously for the NN1 model. This ensures a consistent and structured input pipeline for evaluation and adversarial testing.

In [ ]:
if mtcnn_processing_nn2:
    
    cropped_root_nn2 = os.path.join(test_set_folder, 'cropped_faces_nn2')
    os.makedirs(cropped_root_nn2, exist_ok=True)

    if os.path.isdir(cropped_root_nn2) and os.listdir(cropped_root_nn2):
        logger.info(f"Found existing crops in {cropped_root_nn2}, skipping face cropping.")
    else:
        x_test_aligned_nn2 = []
        y_test_aligned_nn2 = []
        print_done = False
        class_counters = dict()

        for i, (img, y) in enumerate(dataloader):
            class_name = dataset.idx_to_class[y].strip()
            class_dir = os.path.join(cropped_root_nn2, class_name)
            os.makedirs(class_dir, exist_ok=True)

            if class_name not in class_counters:
                class_counters[class_name] = 0
            count = class_counters[class_name]
            class_counters[class_name] += 1
                
            save_path = os.path.join(class_dir, f'img_{count:03d}.jpg')
            
            x_aligned_nn2 = face_detector_nn2(img, return_prob=False, save_path=save_path)

            if x_aligned_nn2 is not None:
                x_aligned_nn2 = nn2_transform(x_aligned_nn2)
                
                x_test_aligned_nn2.append(x_aligned_nn2)
                y_test_aligned_nn2.append(y)

            else:
                logger.info(f'Face not detected in image {i}')

        x_test_aligned_nn2 = torch.stack(x_test_aligned_nn2)
        y_test_aligned_nn2 = torch.tensor(y_test_aligned_nn2)

        torch.save(x_test_aligned_nn2, 'dataset/x_test_aligned_nn2.pt')
        torch.save(y_test_aligned_nn2, 'dataset/y_test_aligned_nn2.pt')

        dataloader_aligned_nn2 = DataLoader(TensorDataset(x_test_aligned_nn2, y_test_aligned_nn2), collate_fn=collate_fn)
        dataloader_aligned_nn2.dataset.idx_to_class = dataset.idx_to_class

        torch.save(dataloader_aligned_nn2, 'dataset/dataloader_aligned_nn2.pt')

The aligned `DataLoader` is loaded from disk, and the test images along with their corresponding labels are extracted into separate tensors. These are then converted into the appropriate format: the image tensor is moved to the CPU and cast to a NumPy array to match the expected input format for subsequent evaluation or adversarial processing.

In [ ]:
# Load the aligned DataLoader
dataloader_aligned_nn2 = torch.load('dataset/dataloader_aligned_nn2.pt', weights_only=False)

# Get the x_test_aligned_nn1 and y_test_aligned_nn1 from the DataLoader
x_test_aligned_nn2, y_test_aligned_nn2 = zip(*[(sample[0], sample[1]) for sample in dataloader_aligned_nn2])

# Create the batches of the tensors of the images and labels
x_test_aligned_nn2 = torch.stack(x_test_aligned_nn2)
y_test_aligned_nn2 = torch.tensor(y_test_aligned_nn2)

x_test_aligned_nn2 = x_test_aligned_nn2.cpu().numpy() 

Using the same evaluation function previously applied to NN1, the model NN2 is now evaluated on the aligned test images via the updated `dataloader_aligned_nn2`. This ensures compatibility between the input resolution and the architecture of NN2. The function returns the `ground truth labels` along with the predicted labels, which are then saved to disk for subsequent analysis.

In [ ]:
# Get the embeddings for the test set
y_true, y_pred_nn2 = evaluate_model(nn2, dataloader_aligned_nn2, LABELS)

torch.save(y_pred_nn2, 'y_pred_nn2.pt')

Now the model is re-evaluated — this time using the predictions made by NN2 — to compute the accuracy on the clean, unperturbed test set. This step allows for a direct comparison with NN1 in terms of generalisation performance.

The accuracy score is calculated and printed, along with a summary of misclassified samples. For clearer visual interpretation, selected identities are displayed using `plot_predicted_images`, helping to illustrate both correct and incorrect predictions through example face images.

In [ ]:
# ————————————————————————————————————————————————————————
#           Valuta NN2 sul test set “clean”
# ————————————————————————————————————————————————————————
from collections import Counter

clean_acc_nn2_folder = os.path.join(results_folder, 'clean_acc_nn2')

img_file_path_nn2 = os.path.join(clean_acc_nn2_folder, 'nn2_clean_predictions.png')

selected_classes = ['Andrea_Bocelli','Gigi_DAlessio','Diego_Abatantuono','Diego_Maradona','Francesco_Totti','Dries_Mertens']

class_to_idx = {v: k for k, v in dataset.idx_to_class.items()}

y_true = torch.load("y_true.pt")

y_pred_nn2 = torch.load("y_pred_nn2.pt")

idx_test_images = []
for name in selected_classes:
    if name in class_to_idx:
        idx_test_images.append(class_to_idx[name])
    else:
        print(f"Name not found: {name}")

correct, incorrect = print_basic_metrics(y_true, y_pred_nn2)
clean_acc_nn2 = accuracy_score(y_true, y_pred_nn2)
print(f"NN2 clean accuracy: {clean_acc_nn2*100:.2f}%")

mismatches = [(t, p) for t, p in zip(y_true, y_pred_nn2) if t != p]
print("\nNot Correctly classified:")
for (t, p), count in Counter(mismatches).most_common(10):
    print(f"❌ {t} → {p}  ({count} volte)")

plot_predicted_images(x_test=x_test_aligned_nn1, y_true=y_true, y_pred=y_pred_nn2, id_test_images=idx_test_images, image_idx=0)

## 3. Attack Transferability Analysis

We now proceed with the analysis of the `transferability of adversarial attacks` originally crafted against the baseline face recognition model (`NN1`), in order to evaluate their effectiveness when applied to an alternative architecture (`NN2`). This assessment is central to understanding whether adversarial perturbations can generalise across different model implementations trained on the same task.

In this experiment, we reuse adversarial examples that were generated using `NN1 wrapped with the ART PyTorchClassifier interface` for generating attacks using the Adversarial Robustness Toolbox (ART). These adversarial inputs are then applied to NN2 without any re-generation or adaptation, simulating a grey-box scenario in which the attacker has full access to the surrogate model (NN1) but no access to the internal parameters of the target model (NN2).

The evaluation pipeline begins by loading the pre-aligned test dataset, `dataloader_aligned_nn1.pt`, from which image and label tensors are extracted and converted into NumPy arrays to ensure compatibility with the ART (Adversarial Robustness Toolbox) interface.

All attack types generated for NN1 are considered in this analysis, with the sole exception of `DeepFool`. This particular attack is excluded as it failed to yield valid adversarial samples under the operational constraints defined for NN1.

In [ ]:
import torch.nn as nn
import torch.optim as optim

classifier_nn1 = PyTorchClassifier(
    model=nn1,                                                     # The PyTorch model to use
    clip_values=(-1, 1),                                           # The minimum and maximum values of the input
    loss=nn.CrossEntropyLoss(),                                    # The loss function
    optimizer=optim.Adam(nn1.parameters(), lr=0.01),               # The optimizer
    input_shape=(3, 160, 160),                                     # The shape of the input
    nb_classes=LABELS.size,                                        # The number of classes
    device_type='cuda' if torch.cuda.is_available() else 'cpu'
)

attack_folder_nn2 = os.path.join(results_folder, 'attack_results_nn2')

# Load the aligned DataLoader
dataloader_aligned_nn1 = torch.load('dataset/dataloader_aligned_nn1.pt', weights_only=False)

# Get the x_test_aligned_nn1 and y_test_aligned_nn1 from the DataLoader
x_test_aligned_nn1, y_test_aligned_nn1 = zip(*[(sample[0], sample[1]) for sample in dataloader_aligned_nn1])

# Create the batches of the tensors of the images and labels
x_test_aligned_nn1 = torch.stack(x_test_aligned_nn1)
y_test_aligned_nn1 = torch.tensor(y_test_aligned_nn1)

x_test_aligned_nn1 = x_test_aligned_nn1.cpu().numpy() 

### FGSM (Fast Gradient Sign Method) Adversarial Attack

We begin the analysis of attack transferability with the first technique under consideration: `FGSM (Fast Gradient Sign Method)`. In particular, this section defines the directory structure used to store the results related to the transferability of FGSM adversarial examples. Specifically, these folders will be used to organise the evaluation outputs — including metrics, visualisations, and comparative plots — for adversarial inputs initially crafted against the baseline model (NN1) and subsequently tested on the alternative model (NN2).

In [ ]:
from art.attacks.evasion import FastGradientMethod

#fgsm_adv_folder_nn2 = os.path.join(adversarial_folder, 'FGSM_nn2')
#os.makedirs(fgsm_adv_folder_nn2, exist_ok=True)
fgsm_results_folder_nn2 = os.path.join(attack_folder_nn2, 'FGSM_results')
os.makedirs(fgsm_results_folder_nn2, exist_ok=True)

fgsm_error_g_nn2_folder = os.path.join(fgsm_results_folder_nn2, 'FGSM_Error_Generic')
os.makedirs(fgsm_error_g_nn2_folder, exist_ok=True)
fgsm_error_s_nn2_folder = os.path.join(fgsm_results_folder_nn2, 'FGSM_Error_Specific')
os.makedirs(fgsm_error_s_nn2_folder, exist_ok=True)

fgsm_img_path_transfer = os.path.join(fgsm_error_g_nn2_folder, "fgsm_error_generic_transfer_plot.png")
fgsm_transfer_sec_img_g_nn2 = os.path.join(fgsm_error_g_nn2_folder, "fgsm_error_generic_transfer_sec_plot.png")
fgsm_transfer_img_s_nn2 = os.path.join(fgsm_error_s_nn2_folder, "fgsm_error_specific_transfer_plot.png")
fgsm_sec_img_s_nn2 = os.path.join(fgsm_error_s_nn2_folder, "fgsm_error_specific_transfer_sec_plot.png")

y_true = torch.load('y_true.pt')

#### Error Generic

Starting from the `Error Generic` configuration, we begin by evaluating the transferability of adversarial examples generated via FGSM. Specifically, in this section, we load the adversarial samples previously crafted against NN1 and, through the use of the custom `build_dataloader_from_adversarial` function, we prepare them for evaluation on NN2.

This function adapts the adversarial inputs to the preprocessing requirements of NN2 — including resizing, colour channel reordering, and mean subtraction — and constructs a `PyTorch DataLoader` suitable for batch evaluation.

The adversarial samples are then passed through NN2, and predictions are compared to the original ground-truth labels to assess the impact of cross-model perturbations.

In [ ]:
x_test_adv_fgsm_g_from_nn1 = torch.load('attacks/FGSM_nn1/x_test_adv_fgsm_g.pt')
x_test_adv_fgsm_g_converted = conversion_nn1_to_nn2(x_test_adv_fgsm_g_from_nn1)

dataloader_adv = build_dataloader_from_adversarial(
    x_adv=x_test_adv_fgsm_g_converted,
    y_true=y_true,
    class_to_idx=class_to_idx,
    idx_to_class=dataset.idx_to_class
)

y_true_adv, y_pred_transfer_nn2 = evaluate_model(nn2, dataloader_adv, LABELS)

fgsm_epsilon = 0.1

print("=== FGSM Transfer Evaluation: NN1 → NN2 ===")
print(f"ε = {fgsm_epsilon}\n")

correct, incorrect = print_basic_metrics(y_true_adv, y_pred_transfer_nn2, x_test_aligned_nn1, x_test_adv_fgsm_g_from_nn1)

acc = accuracy_score(y_true_adv, y_pred_transfer_nn2)
print(f"Transfer Accuracy NN2 (from NN1 adv): {acc*100:.2f}%")

mismatches = [(t, p) for t, p in zip(y_true_adv, y_pred_transfer_nn2) if t != p]
print("\n❌ Most frequent misclassifications:")
for (t, p), count in Counter(mismatches).most_common(10):
    print(f"{t} → {p}  ({count}x)")

plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_adv_fgsm_g_from_nn1,
    y_true=y_true_adv,
    y_pred=y_pred_transfer_nn2,
    y_pred_adv=y_pred_transfer_nn2,
    title=f"FGSM Transfer NN1 → NN2 (ε = {fgsm_epsilon})",
    id_test_images=idx_test_images,
    image_idx=0
)

We now proceed to `generate a series of FGSM adversarial attacks under the Error Generic setting` by systematically `varying` the value of the perturbation parameter, `epsilon (ε)`. This extends the previous analysis, where evaluation was performed using a single fixed ε, by exploring how different perturbation magnitudes affect the transferability of attacks from NN1 to NN2.

The results across all tested ε values are collected and plotted in the form of a `Security Evaluation Curve (SEC)`. This curve provides a clear visual representation of NN2's robustness when exposed to perturbations of increasing strength originally tailored for NN1.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curve: FGSM (Error Generic) NN1 → NN2
# ──────────────────────────────────────────────────────────────────────────────

# Valori di epsilon da testare
epsilons = [0.01, 0.02, 0.03, 0.05, 0.1]

accuracies = [0.98]

print("=== Security Evaluation Curve: FGSM (Error Generic) NN1 → NN2 ===\n")
for eps in epsilons:
    print(f"\n→ Generating adversarial examples with ε = {eps:.3f}")
    
    # Genera attacchi FGSM su NN1
    fgsm = FastGradientMethod(estimator=classifier_nn1, eps=eps)
    x_adv_nn1 = fgsm.generate(x=x_test_aligned_nn1)
    x_adv_converted = conversion_nn1_to_nn2(x_adv_nn1)
    
    # Prepara dataloader per NN2 (trasformazioni incluse)
    dataloader_adv_nn2 = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )

    # Valutazione con NN2
    y_true_eval, y_pred_adv_nn2 = evaluate_model(nn2, dataloader_adv_nn2, LABELS)

    acc = accuracy_score(y_true_eval, y_pred_adv_nn2)
    accuracies.append(acc)

    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_adv_nn2, x_test_aligned_nn1, x_adv_nn1)

# Plot finale
fig, ax = plt.subplots(figsize=(8, 5))
epsilons = [0] + epsilons
plot_multiple_sec_curves(
    ax=ax,
    curves=[(epsilons, accuracies, 'o', 'crimson', 'NN2')],
    title="FGSM Error Generic Transfer NN1 → NN2 - Security Evaluation Curve",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

plt.tight_layout()
plt.savefig(fgsm_transfer_sec_img_g_nn2, bbox_inches='tight')
plt.show()

#### Error Specific

We now proceed with the `Error Specific case for FGSM`, and once again load the adversarial samples previously generated on the NN1 model. These examples were crafted to mislead the classifier into predicting a predefined target identity — in this case, `Fernando_Torres` — regardless of the true label of the input image.

In [ ]:
fgsm_epsilon = 0.1

# Name of the target class
target_name = 'Fernando_Torres'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

As in the previous transferability evaluations, the adversarial inputs are passed through the `build_dataloader_from_adversarial` function to ensure they are preprocessed appropriately for NN2. Once the adapted `DataLoader` is ready, predictions from NN2 are collected and compared to both the original ground-truth labels and the intended target class.

In [ ]:
# Carica dati adversarial generati su NN1
x_test_adv_fgsm_s_from_nn1 = torch.load('attacks/FGSM_nn1/x_test_adv_fgsm_s.pt')
x_test_adv_fgsm_s_converted = conversion_nn1_to_nn2(x_test_adv_fgsm_s_from_nn1)

# Costruzione DataLoader per NN2
dataloader_adv = build_dataloader_from_adversarial(
    x_adv=x_test_adv_fgsm_s_converted,
    y_true=y_true,
    class_to_idx=class_to_idx,
    idx_to_class=dataset.idx_to_class
)

# Valutazione
y_true_adv, y_pred_transfer_s_nn2 = evaluate_model(nn2, dataloader_adv, LABELS)

# Metriche
acc_transfer_s = accuracy_score(y_true_adv, y_pred_transfer_s_nn2)
sr_transfer_s  = (np.array(y_pred_transfer_s_nn2) == target_name).mean() * 100

print("=== FGSM Error Specific Transfer Evaluation: NN1 → NN2 ===")
print(f"ε = {eps}, target = {target_name}")
print(f"Accuracy         : {acc_transfer_s*100:.2f}%")
print(f"Targeted success : {sr_transfer_s:.2f}%\n")

correct, incorrect = print_basic_metrics(y_true_adv, y_pred_transfer_s_nn2, x_test_aligned_nn1, x_test_adv_fgsm_s_from_nn1)

# Plot con immagini adversarial ancora in [-1, 1]
plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_adv_fgsm_s_from_nn1,
    y_true=y_true,
    y_pred=y_pred_transfer_s_nn2,
    y_pred_adv=y_pred_transfer_s_nn2,
    title=f"FGSM Error Specific Transfer NN1 → NN2 (ε = {eps}, target = {target_name})",
    id_test_images=idx_test_images,
    image_idx=0
)

Continuing with the evaluation approach adopted previously for NN1, we now analyse the `Error Specific case for FGSM on NN2`, investigating how the targeted success rate is affected by `variations in the attack parameter epsilon` (`ε`). For each ε value, adversarial examples are generated on NN1 with a fixed target identity, then transferred and evaluated on NN2.

The `Targeted Success Rate` — defined as the proportion of adversarial inputs that successfully induce NN2 to misclassify samples as the intended target class — is computed for each perturbation level. This enables us to build a `Security Evaluation Curve (SEC)` for the error-specific scenario, quantifying the extent to which NN2 is vulnerable to targeted attacks crafted on a different model.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curve: FGSM (Error Specific) NN1 → NN2
# ──────────────────────────────────────────────────────────────────────────────

epsilons = [0.01, 0.02, 0.03, 0.05, 0.1]

accuracies = [0.98]
success_rates = [0.01]

print("=== Security Evaluation Curve: FGSM (Error Specific) NN1 → NN2 ===\n")
for eps in epsilons:
    print(f"\n→ Generating adversarial examples with ε = {eps:.3f}")
    
    fgsm = FastGradientMethod(estimator=classifier_nn1, eps=eps, targeted=True)
    x_adv_nn1 = fgsm.generate(x_test_aligned_nn1, one_hot_targeted_label)
    x_adv_converted = conversion_nn1_to_nn2(x_adv_nn1)
    
    dataloader_adv = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn2, dataloader_adv, LABELS)

    sr = (np.array(y_pred) == target_name).mean()
    success_rates.append(sr)
    
    acc = accuracy_score(y_true, y_pred)
    accuracies.append(acc)
    print(f"   Accuracy All:  {acc*100:.2f}%")

    print(f"   Targeted Success Rate: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv_nn1)

fig, ax = plt.subplots(figsize=(8, 5))
epsilons = [0] + epsilons
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (epsilons, success_rates, 'o', 'crimson', 'Success Rate (Targeted)'),
        (epsilons, accuracies, 's', 'blue', 'Accuracy NN2 (All Classes)')
    ],
    title="FGSM Error Specific Transfer NN1 → NN2 - Security Evaluation Curve",
    xlabel="Epsilon",
    ylabel="Metric"
)

plt.tight_layout()
plt.savefig(fgsm_sec_img_s_nn2, bbox_inches='tight')
plt.show()

### BIM (Basic Iterative Method) Adversarial Attack 

We now proceed by evaluating the `Basic Iterative Method (BIM)` attack, setting up the required directory structure to store all results related to the transferability assessment of this attack.

Specifically, as done for almost all the attacks, two subfolders are created to distinguish between the evaluation of Error Generic and Error Specific scenarios.

In [ ]:
from art.attacks.evasion import BasicIterativeMethod

#bim_adv_folder_nn2 = os.path.join(adversarial_folder, 'BIM_nn2')
#os.makedirs(bim_adv_folder_nn2, exist_ok=True)
bim_results_folder_nn2 = os.path.join(attack_folder_nn2, 'BIM_results')
os.makedirs(bim_results_folder_nn2, exist_ok=True)

bim_error_g_nn2_folder = os.path.join(bim_results_folder_nn2, 'BIM_Error_Generic')
os.makedirs(bim_error_g_nn2_folder, exist_ok=True)
bim_error_s_nn2_folder = os.path.join(bim_results_folder_nn2, "BIM_Error_Specific")
os.makedirs(bim_error_s_nn2_folder, exist_ok=True)

bim_img_path_transfer = os.path.join(bim_error_g_nn2_folder, "bim_error_generic_transfer_plot.png")
bim_transfer_sec_img_g_nn2 = os.path.join(bim_error_g_nn2_folder, "bim_error_generic_transfer_sec_plot.png")
bim_transfer_img_s_nn2 = os.path.join(bim_error_s_nn2_folder, "bim_error_specific_transfer_plot.png")
bim_sec_img_s_nn2 = os.path.join(bim_error_s_nn2_folder, "bim_error_specific_transfer_sec_plot.png")

y_true = torch.load('y_true.pt')

#### Error Generic

We now evaluate the transferability of adversarial examples generated on NN1 using the `Error Generic setting`.

The adversarial samples (`x_test_adv_bim_g.pt`) previously crafted against NN1 are loaded and passed through the `build_dataloader_from_adversarial` function to apply the necessary preprocessing required by NN2. This ensures consistency in the input format and enables a reliable evaluation of cross-model robustness. NN2 is then evaluated on these adversarial inputs.

In [ ]:
# Caricamento adversarial da attacco BIM su NN1
x_test_adv_bim_g_from_nn1 = torch.load('attacks/BIM_nn1/x_test_adv_bim_g.pt')
x_test_adv_bim_g_converted = conversion_nn1_to_nn2(x_test_adv_bim_g_from_nn1)

# Costruzione dataloader da immagini adversarial e etichette
dataloader_adv = build_dataloader_from_adversarial(
    x_adv=x_test_adv_bim_g_converted,
    y_true=y_true,
    class_to_idx=class_to_idx,
    idx_to_class=dataset.idx_to_class
)

# Predizione NN2 su adversarial da NN1
y_true_bim, y_pred_transfer_nn2 = evaluate_model(nn2, dataloader_adv, LABELS)

# Parametri attacco
bim_eps       = 0.03
bim_eps_step  = 0.01
bim_max_iter  = 5

# Log dei risultati
print("=== BIM Transfer Evaluation: NN1 → NN2 ===")
print(f"ε = {bim_eps}, ε_step = {bim_eps_step}, iter = {bim_max_iter}\n")

correct, incorrect = print_basic_metrics(y_true_bim, y_pred_transfer_nn2, x_test_aligned_nn1, x_test_adv_bim_g_from_nn1)

acc_bim_transfer = accuracy_score(y_true_bim, y_pred_transfer_nn2)
print(f"Transfer Accuracy NN2 (from NN1 BIM adv): {acc_bim_transfer*100:.2f}%")

# Errori più frequenti
mismatches_bim = [(t, p) for t, p in zip(y_true_bim, y_pred_transfer_nn2) if t != p]
print("\n❌ Most frequent misclassifications:")
for (t, p), count in Counter(mismatches_bim).most_common(10):
    print(f"{t} → {p}  ({count}x)")

# Visualizzazione risultati
plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_adv_bim_g_from_nn1,
    y_true=y_true,
    y_pred=y_pred_transfer_nn2,
    y_pred_adv=y_pred_transfer_nn2,
    title=f"BIM Transfer NN1 → NN2 (ε = {bim_eps}, step = {bim_eps_step}, iter = {bim_max_iter})",
    id_test_images=idx_test_images,
    image_idx=0
)

As done previously for FGSM, we continue the evaluation of attack transferability by generating `Security Evaluation Curves (SECs)` for BIM in the Error Generic setting. The goal is always the same, to analyse how different attack parameters influence the effectiveness of adversarial examples when transferred from NN1 to NN2.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curves - BIM Transfer (Error Generic) NN1 → NN2
# ──────────────────────────────────────────────────────────────────────────────

eps_values       = [0.01, 0.03, 0.05, 0.1]
eps_step_values  = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values  = [1, 3, 5, 10, 15]

eps              = 0.03
eps_step         = 0.01
max_iter         = 3

curve_colors = ['crimson', 'darkorange', 'seagreen']

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
accuracies_eps  = [0.98]

print("=== SEC Transfer: BIM vs ε ===")
for epsilon in eps_values:
    print(f"\n→ Generating adversarial examples eps = {epsilon:.3f}, eps_step = {eps_step:.3f}, max_iter = {max_iter}")
    bim = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=epsilon,
        eps_step=eps_step,
        max_iter=max_iter
    )
    x_adv = bim.generate(x=x_test_aligned_nn1)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)

    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    accuracies_eps.append(acc)
    print(f"   Transfer Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(eps_values, accuracies_eps, 'o', curve_colors[0], 'NN2')],
    title=f"Accuracy vs Epsilon (ε_step={eps_step}, iter={max_iter})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
accuracies_step = [0.98]

print("\n=== SEC Transfer: BIM vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples eps_step = {step:.3f}, eps = {eps:.3f}, max_iter = {max_iter}")
    bim = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=step,
        max_iter=max_iter
    )
    x_adv = bim.generate(x=x_test_aligned_nn1)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)

    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    accuracies_step.append(acc)
    print(f"   Transfer Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(eps_step_values, accuracies_step, 's', curve_colors[1], 'NN2')],
    title=f"Accuracy vs Epsilon Step (ε={eps}, iter={max_iter})",
    xlabel="ε step",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
accuracies_iter = [0.98]

print("\n=== SEC Transfer: BIM vs max_iter ===")
for n_iter in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {n_iter}, eps = {eps:.3f}, eps_step = {eps_step:.3f}")
    bim = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=eps_step,
        max_iter=n_iter
    )
    x_adv = bim.generate(x=x_test_aligned_nn1)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)
    
    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    accuracies_iter.append(acc)
    print(f"   Transfer Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(max_iter_values, accuracies_iter, '^', curve_colors[2], 'NN2')],
    title=f"Accuracy vs Max Iterations with (ε={eps}, ε_step={eps_step})",
    xlabel="Max Iterations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.savefig(bim_transfer_sec_img_g_nn2, bbox_inches='tight')
plt.show()

#### Error Specific

We now proceed, as usual, with the `Error Specific configuration` of the BIM attack. As done previously, we begin by selecting a specific target identity — in this case, `Fernando_Torres` — and use it as the fixed target class for the attack.

In [ ]:
bim_eps       = 0.03
bim_eps_step  = 0.01
bim_max_iter  = 3

# Name of the target class
target_name = 'Fernando_Torres'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

Adversarial examples, previously generated on NN1 using specific parameters (`ε = 0.03`, `ε_step = 0.01`, `max_iter = 5`), are loaded and evaluated on NN2 after appropriate preprocessing.

In [ ]:
# Carica dati adversarial generati su NN1 (BIM targeted)
x_test_adv_bim_s_from_nn1 = torch.load('attacks/BIM_nn1/x_test_adv_bim_s.pt')
x_test_adv_bim_s_converted = conversion_nn1_to_nn2(x_test_adv_bim_s_from_nn1)

# Costruzione DataLoader per NN2
dataloader_adv = build_dataloader_from_adversarial(
    x_adv=x_test_adv_bim_s_converted,
    y_true=y_true,
    class_to_idx=class_to_idx,
    idx_to_class=dataset.idx_to_class
)

# Valutazione
y_true_adv, y_pred_transfer_s_nn2 = evaluate_model(nn2, dataloader_adv, LABELS)

# Metriche
acc_transfer_s = accuracy_score(y_true_adv, y_pred_transfer_s_nn2)
sr_transfer_s  = (np.array(y_pred_transfer_s_nn2) == target_name).mean() * 100

print("=== BIM Error Specific Transfer Evaluation: NN1 → NN2 ===")
print(f"ε = {bim_eps}, ε_step = {bim_eps_step}, max_iter = {bim_max_iter}, target = {target_name}")
print(f"Accuracy         : {acc_transfer_s*100:.2f}%")
print(f"Targeted success : {sr_transfer_s:.2f}%\n")

correct, incorrect = print_basic_metrics(
    y_true_adv, 
    y_pred_transfer_s_nn2, 
    x_orig=x_test_aligned_nn1, 
    x_adv=x_test_adv_bim_s_from_nn1
)

# Plot predizioni adversarial
plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_adv_bim_s_from_nn1,
    y_true=y_true_adv,
    y_pred=y_pred_transfer_s_nn2,
    y_pred_adv=y_pred_transfer_s_nn2,
    title=f"BIM Error Specific Transfer NN1 → NN2 (ε = {bim_eps}, ε_step = {bim_eps_step}, max_iter = {bim_max_iter}, target = {target_name})",
    id_test_images=idx_test_images,
    image_idx=0,
    save_path=bim_transfer_img_s_nn2
)

We complete the evaluation of the BIM attack in the Error Specific setting by generating `Security Evaluation Curves (SECs)` for the NN1 → NN2 transfer scenario. As in the previous analyses, the goal is to understand how different attack parameters influence both the transfer accuracy and the targeted success rate — the latter representing the proportion of adversarial inputs misclassified as the predefined target identity (`Fernando_Torres`).

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curves - BIM Transfer (Error Specific) NN1 → NN2
# ──────────────────────────────────────────────────────────────────────────────

eps_values      = [0.01, 0.03, 0.05, 0.1]
eps_step_values = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values = [1, 3, 5, 10, 15]

eps             = 0.03
eps_step        = 0.01
max_iter        = 3

curve_colors    = ['crimson', 'blue']

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
acc_eps         = [0.98]
sr_eps          = [0.01]

print("=== SEC Transfer: BIM vs ε ===")
for epsilon in eps_values:
    print(f"\n→ Generating adversarial examples eps = {epsilon:.3f}, eps_step = {eps_step}, iter = {max_iter}")
    attack = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=epsilon,
        eps_step=eps_step,
        max_iter=max_iter,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)
    
    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )
    y_true_eval, y_pred_nn2 = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred_nn2)
    sr  = (np.array(y_pred_nn2) == target_name).mean()
    acc_eps.append(acc)
    sr_eps.append(sr)
    print(f"   Transfer Accuracy: {acc*100:.2f}% | Targeted Success: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_nn2, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_values, acc_eps, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_values, sr_eps,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"ε vs Accuracy / Targeted Accuracy\n(ε_step={eps_step}, iter={max_iter})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
acc_step        = [0.98]
sr_step         = [0.01]

print("\n=== SEC Transfer: BIM vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples eps_step = {step:.3f}, eps = {eps}, iter = {max_iter}")
    attack = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=step,
        max_iter=max_iter,
        targeted=True
    )

    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)
    
    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred_nn2 = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred_nn2)
    sr  = (np.array(y_pred_nn2) == target_name).mean()
    acc_step.append(acc)
    sr_step.append(sr)
    print(f"   Transfer Accuracy: {acc*100:.2f}% | Targeted Success: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_nn2, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_step_values, acc_step, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_step_values, sr_step,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Epsilon Step vs Accuracy / Targeted Accuracy\n(ε={eps}, iter={max_iter})",
    xlabel="Epsilon Step",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
acc_iter    = [0.98]
sr_iter     = [0.01]

print("\n=== SEC Transfer: BIM vs max_iter ===")
for it in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {it}, eps = {eps}, eps_step = {eps_step}")
    attack = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=eps_step,
        max_iter=it,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)

    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )
    y_true_eval, y_pred_nn2 = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred_nn2)

    sr  = (np.array(y_pred_nn2) == target_name).mean()
    acc_iter.append(acc)
    sr_iter.append(sr)
    print(f"   Transfer Accuracy: {acc*100:.2f}% | Targeted Success: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_nn2, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (max_iter_values, acc_iter, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (max_iter_values, sr_iter,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Max Iterations vs Accuracy / Targeted Accuracy\n(ε={eps}, ε_step={eps_step})",
    xlabel="Max Iterations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.savefig(bim_sec_img_s_nn2, bbox_inches='tight')
plt.show()

### PGD (Projected Gradient Descent) Adversarial Attack

Now procede to sets up the necessary directory structure and file paths for managing the results of `Projected Gradient Descent (PGD) adversarial attacks`.

In [ ]:
from art.attacks.evasion import ProjectedGradientDescent

#pgd_adv_folder_nn2 = os.path.join(adversarial_folder, 'PGD_nn2')
#os.makedirs(pgd_adv_folder_nn2, exist_ok=True)
pgd_results_folder_nn2 = os.path.join(attack_folder_nn2, 'PGD_results')
os.makedirs(pgd_results_folder_nn2, exist_ok=True)

pgd_error_g_nn2_folder = os.path.join(pgd_results_folder_nn2, 'PGD_Error_Generic')
os.makedirs(pgd_error_g_nn2_folder, exist_ok=True)
pgd_error_s_nn2_folder = os.path.join(pgd_results_folder_nn2, 'PGD_Error_Specific')
os.makedirs(pgd_error_s_nn2_folder, exist_ok=True)

pgd_img_path_transfer = os.path.join(pgd_error_g_nn2_folder, "pgd_error_generic_transfer_plot.png")
pgd_sec_img_g_nn2 = os.path.join(pgd_error_g_nn2_folder, "pgd_error_generic_transfer_sec_plot.png")
pgd_transfer_img_s_nn2 = os.path.join(pgd_error_s_nn2_folder, "pgd_error_specific_transfer_plot.png")
pgd_sec_img_s_nn2 = os.path.join(pgd_error_s_nn2_folder, "pgd_error_specific_transfer_sec_plot.png")

y_true = torch.load('y_true.pt')

#### Error Generic

In this phase, we evaluate the transferability of adversarial examples generated using the `Projected Gradient Descent (PGD)` attack on model NN1, by assessing their impact on model NN2 under the `Error Generic setting`.

Previously crafted adversarial samples (`x_test_adv_pgd_g.pt`) are loaded and processed using the `build_dataloader_from_adversarial` function. This ensures that the input format matches NN2’s expectations, maintaining consistency and enabling a reliable cross-model robustness evaluation.

In [ ]:
x_test_adv_pgd_g_from_nn1 = torch.load('attacks/PGD_nn1/x_test_adv_pgd_g.pt')
x_test_adv_pgd_g_converted = conversion_nn1_to_nn2(x_test_adv_pgd_g_from_nn1)

dataloader_adv = build_dataloader_from_adversarial(
    x_adv=x_test_adv_pgd_g_converted,
    y_true=y_true,
    class_to_idx=class_to_idx,
    idx_to_class=dataset.idx_to_class
)

y_true_eval, y_pred_transfer_nn2 = evaluate_model(nn2, dataloader_adv, LABELS)

pgd_eps         = 0.03
pgd_eps_step    = 0.01
pgd_max_iter    = 3
num_random_init = 5

print("=== PGD Error Generic Transfer Evaluation: NN1 → NN2 ===")
print(f"ε = {pgd_eps}, ε_step = {pgd_eps_step}, iter = {pgd_max_iter}, init = {num_random_init}\n")

# Metriche
correct, incorrect = print_basic_metrics(y_true_eval, y_pred_transfer_nn2, x_test_aligned_nn1, x_test_adv_pgd_g_from_nn1)

acc_transfer_pgd = accuracy_score(y_true_eval, y_pred_transfer_nn2)
print(f"Transfer Accuracy NN2 (from NN1 PGD adv): {acc_transfer_pgd*100:.2f}%")

# Errori frequenti
mismatches = [(t, p) for t, p in zip(y_true_eval, y_pred_transfer_nn2) if t != p]
print("\n❌ Most frequent misclassifications:")
for (t, p), count in Counter(mismatches).most_common(10):
    print(f"{t} → {p}  ({count}x)")

# Visualizzazione
plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_adv_pgd_g_from_nn1,
    y_true=y_true_eval,
    y_pred=y_pred_transfer_nn2,
    y_pred_adv=y_pred_transfer_nn2,
    title=f"PGD Transfer NN1 → NN2 (ε = {pgd_eps}, step = {pgd_eps_step}, iter = {pgd_max_iter})",
    id_test_images=idx_test_images,
    image_idx=0,
    save_path=pgd_img_path_transfer
)

As done previously for FGSM and BIM, we continue the assessment of adversarial transferability by generating the `Security Evaluation Curves (SECs) for the Projected Gradient Descent (PGD)` attack under the `Error Generic setting`. The objective remains to investigate how varying attack parameters influences the transfer success of adversarial examples generated on NN1 when applied to NN2.

A series of experiments is carried out, where the following PGD parameters are varied one at a time:

- `ε` (perturbation bound)

- `ε_step` (step size)

- `max_iter` (number of iterations)

- `num_random_init` (number of random initialisations)

For each variation, adversarial examples are generated on NN1, transferred to NN2, and evaluated. The accuracy of NN2 on each adversarial batch is plotted accordingly.

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - PGD (Error Generic) NN1 → NN2 
# —————————————————————————————————————————————————————————————

eps_values       = [0.01, 0.03, 0.05, 0.10]
eps_step_values  = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values  = [1, 3, 5, 10, 15]
num_init_values  = [1, 3, 5]

pgd_eps          = 0.03
pgd_eps_step     = 0.01
pgd_max_iter     = 3
num_random_init  = 5

curve_colors = ['crimson', 'darkorange', 'seagreen', 'royalblue']

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
acc_eps = [0.98]

print("=== SEC Transfer: PGD vs ε ===")
for eps in eps_values:
    print(f"\n→ Generating adversarial examples with eps = {eps:.3f}, eps_step = {pgd_eps_step:.3f}, max_iter = {pgd_max_iter}, init = {num_random_init}")
    pgd = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=pgd_eps_step,
        max_iter=pgd_max_iter,
        num_random_init=num_random_init,
        targeted=False
    )
    x_adv = pgd.generate(x=x_test_aligned_nn1)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)

    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    acc_eps.append(acc)

    print(f"   Transfer Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(eps_values, acc_eps, 'o', curve_colors[0], 'NN2')],
    title=f"Accuracy vs Epsilon (ε_step={pgd_eps_step}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
acc_step = [0.98]

print("\n=== SEC Transfer: PGD vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples with eps_step = {step:.3f}, eps = {pgd_eps:.3f}, max_iter = {pgd_max_iter}, init = {num_random_init}")
    pgd = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=step,
        max_iter=pgd_max_iter,
        num_random_init=num_random_init,
        targeted=False
    )

    x_adv = pgd.generate(x=x_test_aligned_nn1)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)

    dataloader = build_dataloader_from_adversarial(x_adv_converted, y_true, class_to_idx, dataset.idx_to_class)

    y_true_eval, y_pred = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    acc_step.append(acc)
    print(f"ε_step = {step:.3f} | Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(eps_step_values, acc_step, 's', curve_colors[1], 'NN2')],
    title=f"Accuracy vs Epsilon Step (ε={pgd_eps}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsilon Step",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
acc_iter = [0.98]

print("\n=== SEC Transfer: PGD vs max_iter ===")
for it in max_iter_values:
    print(f"\n→ Generating adversarial examples with max_iter = {it}, eps = {pgd_eps:.3f}, eps_step = {pgd_eps_step:.3f}, init = {num_random_init}")
    pgd = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=pgd_eps_step,
        max_iter=it,
        num_random_init=num_random_init,
        targeted=False
    )
    
    x_adv = pgd.generate(x=x_test_aligned_nn1)
    
    x_adv_converted = conversion_nn1_to_nn2(x_adv)
    
    dataloader = build_dataloader_from_adversarial(x_adv_converted, y_true, class_to_idx, dataset.idx_to_class)
    
    y_true_eval, y_pred = evaluate_model(nn2, dataloader, LABELS)
    
    acc = accuracy_score(y_true_eval, y_pred)
    acc_iter.append(acc)
    print(f"max_iter = {it} | Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(max_iter_values, acc_iter, '^', curve_colors[2], 'NN2')],
    title=f"Accuracy vs Max iterations (ε={pgd_eps}, ε_step={pgd_eps_step}, init={num_random_init})",
    xlabel="Max iterations",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 4) Accuracy vs num_random_init
# —————————————————————————————————————————————————————————————
acc_init = [0.98]

print("\n=== SEC Transfer: PGD vs num_random_init ===")
for init in num_init_values:
    print(f"\n→ Generating adversarial examples with num_random_init = {init}, eps = {pgd_eps:.3f}, eps_step = {pgd_eps_step:.3f}, iter = {pgd_max_iter}")
    pgd = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=pgd_eps_step,
        max_iter=pgd_max_iter,
        num_random_init=init,
        targeted=False
    )
    
    x_adv = pgd.generate(x=x_test_aligned_nn1)
    
    x_adv_converted = conversion_nn1_to_nn2(x_adv)
    
    dataloader = build_dataloader_from_adversarial(x_adv_converted, y_true, class_to_idx, dataset.idx_to_class)
    
    y_true_eval, y_pred = evaluate_model(nn2, dataloader, LABELS)
    
    acc = accuracy_score(y_true_eval, y_pred)
    acc_init.append(acc)
    print(f"num_random_init = {init} | Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
num_init_values = [0] + num_init_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(num_init_values, acc_init, 'd', curve_colors[3], 'NN2')],
    title=f"Accuracy vs Random Initializations (ε={pgd_eps}, ε_step={pgd_eps_step}, iter={pgd_max_iter})",
    xlabel="Random Initializations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(pgd_sec_img_g_nn2, bbox_inches='tight')
plt.show()


#### Error Specific

We now proceed, as usual, with the evaluation of adversarial transferability under the `Error Specific configuration` using the `Projected Gradient Descent (PGD)` attack.

As done previously, a fixed target identity is selected — in this case, `Fernando_Torres` — and all adversarial examples are crafted to mislead the model into predicting this specific identity.

In [ ]:
# Name of the target class
target_name = 'Fernando_Torres'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

The adversarial samples, previously generated on NN1 using the following parameters:

- `ε = 0.03`

- `ε_step = 0.01`

- `max_iter = 10`

- `num_random_init = 5`

are now loaded and evaluated on NN2 after applying the appropriate preprocessing to ensure input compatibility.

In [ ]:
# Carica dati adversarial PGD targeted generati su NN1
x_test_adv_pgd_s_from_nn1 = torch.load('attacks/PGD_nn1/x_test_adv_pgd_s.pt')
x_test_adv_pgd_s_converted = conversion_nn1_to_nn2(x_test_adv_pgd_s_from_nn1)

# Costruzione DataLoader
dataloader_adv = build_dataloader_from_adversarial(
    x_adv=x_test_adv_pgd_s_converted,
    y_true=y_true,
    class_to_idx=class_to_idx,
    idx_to_class=dataset.idx_to_class
)

# Valutazione
y_true_adv, y_pred_transfer_s_nn2 = evaluate_model(nn2, dataloader_adv, LABELS)

# Metriche
acc_transfer_s = accuracy_score(y_true_adv, y_pred_transfer_s_nn2)
sr_transfer_s  = (np.array(y_pred_transfer_s_nn2) == target_name).mean() * 100

# Parametri attacco
pgd_eps         = 0.03
pgd_eps_step    = 0.01
pgd_max_iter    = 3
pgd_num_init    = 5

print("=== PGD Error Specific Transfer Evaluation: NN1 → NN2 ===")
print(f"ε = {pgd_eps}, ε_step = {pgd_eps_step}, max_iter = {pgd_max_iter}, num_init = {pgd_num_init}, target = {target_name}")
print(f"Accuracy         : {acc_transfer_s*100:.2f}%")
print(f"Targeted success : {sr_transfer_s:.2f}%\n")

correct, incorrect = print_basic_metrics(y_true_adv, y_pred_transfer_s_nn2, x_test_aligned_nn1, x_test_adv_pgd_s_from_nn1)

# Visualizzazione
plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_adv_pgd_s_from_nn1,
    y_true=y_true_adv,
    y_pred=y_pred_transfer_s_nn2,
    y_pred_adv=y_pred_transfer_s_nn2,
    title=f"PGD Error Specific Transfer NN1 → NN2 (ε = {pgd_eps}, ε_step = {pgd_eps_step}, max_iter = {pgd_max_iter}, num_init = {pgd_num_init}, target = {target_name})",
    id_test_images=idx_test_images,
    image_idx=0,
    save_path=pgd_transfer_img_s_nn2
)

We complete the evaluation of the `Projected Gradient Descent (PGD)` attack in the `Error Specific setting` by generating `Security Evaluation Curves (SECs)` for the NN1 → NN2 transfer scenario. As in previous analyses, the purpose is to examine how different attack parameters influence both the transfer accuracy and the targeted success rate — the latter representing the percentage of adversarial inputs that are misclassified by NN2 as the predefined target identity, `Fernando_Torres`.

The evaluation involves systematically varying one PGD parameter at a time while keeping the others fixed, and observing how these changes affect the behaviour of the transferred attack.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curves - PGD (Error Specific - Targeted) NN1 → NN2
# ──────────────────────────────────────────────────────────────────────────────

eps_values          = [0.01, 0.03, 0.05, 0.10]
eps_step_values     = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values     = [1, 3, 5, 10, 15]
num_init_values     = [1, 3, 5]

pgd_eps             = 0.03
pgd_eps_step        = 0.01
pgd_max_iter        = 3
num_random_init     = 5

curve_colors        = ['crimson', 'blue']

# 1) Accuracy & Targeted Success vs ε
acc_eps = [0.98]
sr_eps = [0.01]

print("=== SEC Transfer: PGD vs ε ===")
for eps in eps_values:
    print(f"\n→ Generating adversarial examples eps = {eps:.3f}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=pgd_eps_step,
        max_iter=pgd_max_iter,
        num_random_init=num_random_init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)

    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )
    y_true_eval, y_pred_nn2 = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred_nn2)
    sr  = (np.array(y_pred_nn2) == target_name).mean()
    acc_eps.append(acc)
    sr_eps.append(sr)
    print(f"   Transfer Accuracy: {acc*100:.2f}% | Targeted Success: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_nn2, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_values, acc_eps, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_values, sr_eps,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Epsilon \n(ε_step={pgd_eps_step}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# 2) Accuracy & Targeted Success vs ε_step
acc_step = [0.98]
sr_step = [0.01]

print("\n=== SEC Transfer: PGD vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples eps_step = {step:.3f}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=step,
        max_iter=pgd_max_iter,
        num_random_init=num_random_init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)

    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )
    y_true_eval, y_pred_nn2 = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred_nn2)
    sr  = (np.array(y_pred_nn2) == target_name).mean()
    acc_step.append(acc)
    sr_step.append(sr)
    print(f"   Transfer Accuracy: {acc*100:.2f}% | Targeted Success: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_nn2, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_step_values, acc_step, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_step_values, sr_step,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Epsilon Step \n(ε={pgd_eps}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsion Step",
    ylabel="Accuracy"
)

# 3) Accuracy & Targeted Success vs max_iter
acc_iter, sr_iter = [0.98], [0.01]

print("\n=== SEC Transfer: PGD vs max_iter ===")
for it in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {it}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=pgd_eps_step,
        max_iter=it,
        num_random_init=num_random_init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)

    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )
    y_true_eval, y_pred_nn2 = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred_nn2)
    sr  = (np.array(y_pred_nn2) == target_name).mean()
    acc_iter.append(acc)
    sr_iter.append(sr)
    print(f"   Transfer Accuracy: {acc*100:.2f}% | Targeted Success: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_nn2, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (max_iter_values, acc_iter, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (max_iter_values, sr_iter,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Max Iterations\n(ε={pgd_eps}, ε_step={pgd_eps_step}, init={num_random_init})",
    xlabel="Max Iterations",
    ylabel="Accuracy"
)

# 4) Accuracy & Targeted Success vs num_random_init
acc_init, sr_init = [0.98], [0.01]

print("\n=== SEC Transfer: PGD vs num_random_init ===")
for init in num_init_values:
    print(f"\n→ Generating adversarial examples num_random_init = {init}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=pgd_eps_step,
        max_iter=pgd_max_iter,
        num_random_init=init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    x_adv_converted = conversion_nn1_to_nn2(x_adv)

    dataloader = build_dataloader_from_adversarial(
        x_adv=x_adv_converted,
        y_true=y_true,
        class_to_idx=class_to_idx,
        idx_to_class=dataset.idx_to_class
    )
    y_true_eval, y_pred_nn2 = evaluate_model(nn2, dataloader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred_nn2)
    sr  = (np.array(y_pred_nn2) == target_name).mean()
    acc_init.append(acc)
    sr_init.append(sr)
    print(f"   Transfer Accuracy: {acc*100:.2f}% | Targeted Success: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_nn2, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
num_init_values = [0] + num_init_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (num_init_values, acc_init, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (num_init_values, sr_init,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Random Initializations\n(ε={pgd_eps}, ε_step={pgd_eps_step}, iter={pgd_max_iter})",
    xlabel="Random Initializations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(pgd_sec_img_s_nn2, bbox_inches='tight')
plt.show()

### Carlini Wagner (CW) $L_{\infty}$ - Adversarial Attack

We conclude the evaluation of the transferability of adversarial attacks crafted on NN1 by considering the `Carlini & Wagner L∞ attack`. In this section, we set up the directory structure necessary to store the results of this final analysis on NN2.

In [ ]:
from art.attacks.evasion import CarliniLInfMethod

#cw_adv_folder_nn2 = os.path.join(adversarial_folder, 'CW_L_inf_nn2')
#os.makedirs(cw_adv_folder_nn2, exist_ok=True)
cw_results_folder_nn2 = os.path.join(attack_folder_nn2, 'CW_results')
cw_error_g_nn2_folder = os.path.join(cw_results_folder_nn2, "CW_Error_Generic_L_inf")
os.makedirs(cw_error_g_nn2_folder, exist_ok=True)

cw_img_path_transfer = os.path.join(cw_error_g_nn2_folder, "cw_linf_error_generic_transfer_plot.png")

y_true = torch.load('y_true.pt')

#### Error Generic

Going on with the transferability evaluation by analysing the behaviour of NN2 when exposed to adversarial examples generated on NN1 using the `Carlini & Wagner L∞ (CW L∞) attack in the Error Generic setting`.

The adversarial samples are loaded and preprocessed through the `build_dataloader_from_adversarial` function to ensure compatibility with NN2’s input format. The model is then evaluated, and the transfer accuracy is computed to assess how many perturbed samples are still classified correctly by NN2.

In [ ]:
x_test_adv_cw_g_from_nn1 = torch.load('attacks/CW_L_inf_nn1/x_test_adv_cw_g.pt')
x_test_adv_cw_g_converted = conversion_nn1_to_nn2(x_test_adv_cw_g_from_nn1)

dataloader_adv = build_dataloader_from_adversarial(
    x_adv=x_test_adv_cw_g_converted,
    y_true=y_true,
    class_to_idx=class_to_idx,
    idx_to_class=dataset.idx_to_class
)

y_true_adv, y_pred_transfer_nn2 = evaluate_model(nn2, dataloader_adv, LABELS)

acc = accuracy_score(y_true_adv, y_pred_transfer_nn2)
print("=== CW L∞ Error Generic Transfer Evaluation: NN1 → NN2 ===")
print(f"Transfer Accuracy NN2: {acc*100:.2f}%")

correct, incorrect = print_basic_metrics(y_true_adv, y_pred_transfer_nn2, x_test_aligned_nn1, x_test_adv_cw_g_from_nn1)

mismatches = [(t, p) for t, p in zip(y_true_adv, y_pred_transfer_nn2) if t != p]
print("\n❌ Most frequent misclassifications:")
for (t, p), count in Counter(mismatches).most_common(10):
    print(f"{t} → {p}  ({count}x)")

plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_adv_cw_g_from_nn1,
    y_true=y_true_adv,
    y_pred=y_pred_transfer_nn2,
    y_pred_adv=y_pred_transfer_nn2,
    title="CW L∞ Transfer NN1 → NN2 (confidence = 0.0, max_iter = 5)",
    id_test_images=idx_test_images,
    image_idx=0,
    save_path=cw_img_path_transfer
)

#### Error Specific

We conclude the transferability evaluation by analysing the Error Specific scenario of the `Carlini & Wagner L∞ attack`. As in previous experiments, we select a fixed target identity — `Fernando_Torres` — and use it to craft targeted adversarial examples on NN1.

In [ ]:
# Name of the target class
target_name = 'Fernando_Torres'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

These samples are then evaluated on NN2 to assess whether the attack can successfully induce targeted misclassifications across models.

In [ ]:
x_test_adv_cw_s_from_nn1 = torch.load('attacks/CW_L_inf_nn1/x_test_adv_cw_s.pt')
x_test_adv_cw_s_converted = conversion_nn1_to_nn2(x_test_adv_cw_s_from_nn1)

dataloader_adv = build_dataloader_from_adversarial(
    x_adv=x_test_adv_cw_s_converted,
    y_true=y_true,
    class_to_idx=class_to_idx,
    idx_to_class=dataset.idx_to_class
)

_, y_pred_transfer_cw_s_nn2 = evaluate_model(nn2, dataloader_adv, LABELS)

acc_transfer_cw_s = accuracy_score(y_true, y_pred_transfer_cw_s_nn2)
sr_transfer_cw_s  = (np.array(y_pred_transfer_cw_s_nn2) == target_name).mean() * 100

cw_target_conf     = 0.5
cw_learning_rate   = 0.01
cw_max_iter        = 7
cw_initial_const   = 1e-5
cw_largest_const   = 1.0
cw_const_factor    = 0.5
cw_decrease_factor = 0.9
cw_batch_size      = 8

print("=== CW L∞ Error Specific Transfer Evaluation: NN1 → NN2 ===")
print(f"confidence = {cw_target_conf}, max_iter = {cw_max_iter}, target = {target_name}")
print(f"Accuracy         : {acc_transfer_cw_s*100:.2f}%")
print(f"Targeted success : {sr_transfer_cw_s:.2f}%\n")

correct, incorrect = print_basic_metrics(
    y_true,
    y_pred_transfer_cw_s_nn2,
    x_orig=x_test_aligned_nn1,
    x_adv=x_test_adv_cw_s_from_nn1
)

cw_transfer_img_s_nn2 = os.path.join(cw_error_g_nn2_folder, f"cw_linf_error_specific_transfer_{target_name}.png")

plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_adv_cw_s_from_nn1,
    y_true=y_true,
    y_pred=y_pred_transfer_cw_s_nn2,
    y_pred_adv=y_pred_transfer_cw_s_nn2,
    title=f"CW L∞ Error Specific Transfer NN1 → NN2 (target_confidence = {cw_target_conf}, learning_rate = {cw_learning_rate}, max_iter = {cw_max_iter}, initial_const = {cw_initial_const}, decrease_factor = {cw_decrease_factor})",
    id_test_images=idx_test_images,
    image_idx=0,
    save_path=cw_transfer_img_s_nn2
)